# 18 — ICH 2.5D Blood-Window with Expanded True Negatives

## Goal

Train a controlled 2.5D ICH segmentation model that combines two strong ideas from the previous experiments:

- **Real neighboring-slice context:** previous, center, and next Blood-window slices are the three input channels.
- **Expanded true negatives:** every slice from a series with zero ground-truth ICH volume is a valid background target, even when that slice has no segmentation JSON.

In ICH-positive series, only slices with valid segmentation masks are used as segmentation targets. Unannotated slices from ICH-positive series may be used as neighboring input images, but they are never treated as negative targets.

The model predicts the six-class segmentation mask of the center slice:

`Background, EDH, SDH, IPH, SAH, IVH`

This notebook uses the same common TRAIN / DEV split logic as Notebooks 16 and 17 and does not evaluate the locked TEST split.


## 1. Environment

In [ ]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg segmentation-models-pytorch

## 2. Imports

In [ ]:
import json
import math
import random
import re
import time
import warnings
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from pydicom.pixels import apply_modality_lut
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import tv_tensors
from torchvision.transforms import v2
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Invalid value for VR UI.*")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 3. Reproducibility and paths

In [ ]:
SEED = 20260918
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TEST = 68
N_DEV = 54
N_SPLIT_TRIALS = 1500

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct"),
    Path("/kaggle/input/iaaa-contest-bct"),
]

OUTPUT_ROOT = Path("/kaggle/working/ich_2d_blood_expanded_true_negatives")
MODELS_DIR = OUTPUT_ROOT / "models"
METRICS_DIR = OUTPUT_ROOT / "metrics"
CACHE_DIR = OUTPUT_ROOT / "cache"
SPLITS_DIR = OUTPUT_ROOT / "splits"
for path in [MODELS_DIR, METRICS_DIR, CACHE_DIR, SPLITS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output root:", OUTPUT_ROOT)

## 4. Experiment configuration

In [ ]:
ICH_CLASSES = ["EDH", "SDH", "IPH", "SAH", "IVH"]
ICH_COLUMNS = [f"V_{name}" for name in ICH_CLASSES]
ORIGINAL_CLASS_ID = {"EDH": 4, "SDH": 3, "IPH": 2, "SAH": 5, "IVH": 1}

BLOOD_WINDOW_CENTER = 45.0
BLOOD_WINDOW_WIDTH = 100.0
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

MODEL_CONFIG = {
    "architecture": "Unet",
    "encoder_name": "efficientnet-b4",
    "encoder_weights": "imagenet",
    "custom_init_checkpoint": None,
}

BATCH_SIZE = 4
NUM_WORKERS = 2
EPOCHS = 30
LEARNING_RATE = 1e-4
LR_SWITCH_EPOCH = 20
LR_AFTER_SWITCH = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 10

SAMPLES_PER_EPOCH = 6000
CATEGORY_WEIGHT = {
    "positive_slice": 1.0,
    "negative_in_positive_series": 1.0,
    "expanded_negative_in_normal_series": 1.0,
}

THRESHOLD_GRID = np.arange(0.20, 0.96, 0.05)
INFERENCE_BATCH_SIZE = 8
HU_CACHE_SIZE = 128
FIXED_CONTINUITY_MIN_RUN = 3
CONTEXT_OFFSETS = (-1, 0, 1)

print("Model config:", MODEL_CONFIG)
print("Blood window:", (BLOOD_WINDOW_CENTER, BLOOD_WINDOW_WIDTH))
print("Samples per epoch:", SAMPLES_PER_EPOCH)

### Easy encoder and pretrained-model switching

Change only `MODEL_CONFIG`.

Examples:

```python
MODEL_CONFIG["encoder_name"] = "efficientnet-b5"
MODEL_CONFIG["encoder_weights"] = "imagenet"
```

or:

```python
MODEL_CONFIG["encoder_weights"] = None
```

For a custom segmentation checkpoint:

```python
MODEL_CONFIG["custom_init_checkpoint"] = "/kaggle/input/.../checkpoint.pth"
```

Only matching tensors are transferred. Do not use a checkpoint that was fine-tuned on data outside the current TRAIN split if you want a clean common-split comparison.

## 5. Validate model configuration

In [ ]:
AVAILABLE_ENCODERS = set(smp.encoders.get_encoder_names())
ARCHITECTURE_BUILDERS = {
    "Unet": smp.Unet,
    "UnetPlusPlus": smp.UnetPlusPlus,
    "FPN": smp.FPN,
    "DeepLabV3Plus": smp.DeepLabV3Plus,
}

if MODEL_CONFIG["architecture"] not in ARCHITECTURE_BUILDERS:
    raise ValueError(f"Unsupported architecture: {MODEL_CONFIG['architecture']}")
if MODEL_CONFIG["encoder_name"] not in AVAILABLE_ENCODERS:
    matches = sorted(name for name in AVAILABLE_ENCODERS if MODEL_CONFIG["encoder_name"].split("-")[0] in name)[:30]
    raise ValueError(f"Encoder {MODEL_CONFIG['encoder_name']} is unavailable. Similar encoders: {matches}")

display(pd.DataFrame([MODEL_CONFIG]))

## 6. Locate the source CT data

In [ ]:
def first_existing(paths):
    return next((path for path in paths if path.exists()), None)

def normalize_series_id(value):
    if pd.isna(value):
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()

DATASET_ROOT = first_existing(DATASET_ROOT_CANDIDATES)
if DATASET_ROOT is None:
    raise FileNotFoundError("CT dataset root was not found.")

DATA_ROOT = first_existing([DATASET_ROOT / "iaaa-contest-bct" / "Data", DATASET_ROOT / "Data"])
if DATA_ROOT is None:
    raise FileNotFoundError("Data directory was not found.")

TRAINING_DIR = DATA_ROOT / "training"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
if not TRAINING_DIR.exists() or not ANNOTATIONS_DIR.exists():
    raise FileNotFoundError("Data/training or Data/annotations was not found.")

TARGETS_PATH = first_existing([
    DATASET_ROOT / "series_targets_df.csv",
    DATASET_ROOT / "iaaa-contest-bct" / "series_targets_df.csv",
    DATA_ROOT / "series_targets_df.csv",
    DATA_ROOT.parent / "series_targets_df.csv",
])
if TARGETS_PATH is None:
    raise FileNotFoundError("series_targets_df.csv was not found.")

print("Dataset root:", DATASET_ROOT)
print("Training DICOMs:", TRAINING_DIR)
print("Annotations:", ANNOTATIONS_DIR)
print("Targets:", TARGETS_PATH)

## 7. Reconstruct the common TRAIN / DEV split

In [ ]:
targets_df = pd.read_csv(TARGETS_PATH).drop(columns=["Unnamed: 0"], errors="ignore").copy()
REQUIRED_TARGET_COLUMNS = ["series_id", "V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm", "triage_class"]
missing_target_columns = [column for column in REQUIRED_TARGET_COLUMNS if column not in targets_df.columns]
if missing_target_columns:
    raise KeyError(f"Missing target columns: {missing_target_columns}")

targets_df["series_id"] = targets_df["series_id"].map(normalize_series_id)
for column in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]:
    targets_df[column] = pd.to_numeric(targets_df[column], errors="raise")
targets_df["triage_class"] = pd.to_numeric(targets_df["triage_class"], errors="raise").astype(int)
targets_df = targets_df.drop_duplicates("series_id").reset_index(drop=True)

if len(targets_df) != 338:
    raise RuntimeError(f"Expected 338 target series, found {len(targets_df)}.")

def add_split_features(dataframe):
    result = dataframe.copy()
    total_ich = result[ICH_COLUMNS].sum(axis=1)
    result["feature_any_ich"] = (total_ich >= 0.1).astype(int)
    result["feature_fracture"] = (result["fracture_prob"] >= 0.5).astype(int)
    for column in ICH_COLUMNS:
        result[f"feature_{column}"] = (result[column] >= 0.1).astype(int)
    result["mls_bin"] = pd.cut(result["MLS_mm"], bins=[-0.01, 1.0, 3.0, 5.0, np.inf], labels=False, include_lowest=True).astype(int)
    for triage_class in [0, 1, 2]:
        result[f"feature_triage_{triage_class}"] = (result["triage_class"] == triage_class).astype(int)
    for mls_bin in [0, 1, 2, 3]:
        result[f"feature_mls_bin_{mls_bin}"] = (result["mls_bin"] == mls_bin).astype(int)
    return result

BALANCE_COLUMNS = [
    "feature_triage_0", "feature_triage_1", "feature_triage_2", "feature_any_ich", "feature_fracture",
    "feature_V_EDH", "feature_V_SDH", "feature_V_IPH", "feature_V_SAH", "feature_V_IVH",
    "feature_mls_bin_0", "feature_mls_bin_1", "feature_mls_bin_2", "feature_mls_bin_3",
]

def choose_balanced_subset(dataframe, subset_size, seed_start, n_trials):
    best = None
    full_prevalence = dataframe[BALANCE_COLUMNS].mean()
    required_positive_columns = [
        "feature_fracture", "feature_V_EDH", "feature_V_SDH", "feature_V_IPH", "feature_V_SAH", "feature_V_IVH",
        "feature_mls_bin_1", "feature_mls_bin_2", "feature_mls_bin_3",
    ]

    for trial_seed in range(seed_start, seed_start + n_trials):
        remaining, subset = train_test_split(dataframe, test_size=int(subset_size), random_state=int(trial_seed), shuffle=True, stratify=dataframe["triage_class"])
        if any(subset[column].sum() == 0 for column in required_positive_columns):
            continue
        if any(remaining[column].sum() == 0 for column in required_positive_columns):
            continue
        subset_prevalence = subset[BALANCE_COLUMNS].mean()
        remaining_prevalence = remaining[BALANCE_COLUMNS].mean()
        score = float((subset_prevalence - full_prevalence).abs().mean() + (remaining_prevalence - full_prevalence).abs().mean())
        if best is None or score < best["score"]:
            best = {"score": score, "seed": trial_seed, "remaining": remaining.copy(), "subset": subset.copy()}

    if best is None:
        raise RuntimeError("Could not reconstruct the common split.")
    return best

split_df = add_split_features(targets_df)
test_split = choose_balanced_subset(split_df, N_TEST, SEED, N_SPLIT_TRIALS)
train_dev_df = test_split["remaining"].copy().reset_index(drop=True)
test_df = test_split["subset"].copy().reset_index(drop=True)
dev_split = choose_balanced_subset(train_dev_df, N_DEV, SEED + N_SPLIT_TRIALS + 1, N_SPLIT_TRIALS)
train_series_df = dev_split["remaining"][REQUIRED_TARGET_COLUMNS].copy().reset_index(drop=True)
dev_series_df = dev_split["subset"][REQUIRED_TARGET_COLUMNS].copy().reset_index(drop=True)

TRAIN_SERIES = set(train_series_df["series_id"])
DEV_SERIES = set(dev_series_df["series_id"])
TEST_SERIES = set(test_df["series_id"])

if TRAIN_SERIES & DEV_SERIES or TRAIN_SERIES & TEST_SERIES or DEV_SERIES & TEST_SERIES:
    raise RuntimeError("Series overlap detected.")
if len(TRAIN_SERIES) != 216 or len(DEV_SERIES) != 54 or len(TEST_SERIES) != 68:
    raise RuntimeError(f"Unexpected split sizes: TRAIN={len(TRAIN_SERIES)}, DEV={len(DEV_SERIES)}, TEST={len(TEST_SERIES)}.")

train_series_df.to_csv(SPLITS_DIR / "common_train_series.csv", index=False)
dev_series_df.to_csv(SPLITS_DIR / "common_dev_series.csv", index=False)

print("TRAIN:", len(TRAIN_SERIES), "| DEV:", len(DEV_SERIES), "| held-out TEST:", len(TEST_SERIES))
print("Locked TEST evaluation is disabled.")

## 8. Build the master slice index

In [ ]:
def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return float(default)

def safe_tuple(value):
    try:
        return tuple(float(item) for item in value)
    except Exception:
        return None

def parse_rle_header(rle):
    if not isinstance(rle, dict) or "shape" not in rle or "counts" not in rle:
        return None
    shape = tuple(int(value) for value in rle["shape"])
    counts = [int(value) for value in rle["counts"]]
    if len(shape) != 2 or len(counts) % 2 != 0 or sum(counts[1::2]) != int(np.prod(shape)):
        return None
    return shape, counts

def read_dicom_header(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    spacing = safe_tuple(getattr(ds, "PixelSpacing", None))
    orientation = safe_tuple(getattr(ds, "ImageOrientationPatient", None))
    position = safe_tuple(getattr(ds, "ImagePositionPatient", None))
    position_scalar = np.nan

    if orientation is not None and position is not None and len(orientation) >= 6 and len(position) >= 3:
        row_cosine = np.asarray(orientation[:3], dtype=np.float64)
        col_cosine = np.asarray(orientation[3:6], dtype=np.float64)
        normal = np.cross(row_cosine, col_cosine)
        position_scalar = float(np.dot(np.asarray(position[:3], dtype=np.float64), normal))

    return {
        "sop_uid": str(getattr(ds, "SOPInstanceUID", path.stem)),
        "row_spacing": spacing[0] if spacing is not None else np.nan,
        "col_spacing": spacing[1] if spacing is not None else np.nan,
        "instance_number": safe_float(getattr(ds, "InstanceNumber", np.nan)),
        "position_scalar": position_scalar,
        "slice_thickness": abs(safe_float(getattr(ds, "SliceThickness", np.nan))),
        "spacing_between_slices": abs(safe_float(getattr(ds, "SpacingBetweenSlices", np.nan))),
    }

MASTER_INDEX_PATH = CACHE_DIR / "master_slice_index.pkl"

if MASTER_INDEX_PATH.exists():
    master_df = pd.read_pickle(MASTER_INDEX_PATH)
else:
    rows = []
    series_dirs = sorted([path for path in TRAINING_DIR.iterdir() if path.is_dir()], key=lambda path: int(path.name))

    for series_dir in tqdm(series_dirs, desc="Building master slice index"):
        series_id = normalize_series_id(series_dir.name)
        split_name = "TRAIN" if series_id in TRAIN_SERIES else "DEV" if series_id in DEV_SERIES else "TEST"

        for dicom_path in sorted(series_dir.glob("*.dcm")):
            header = read_dicom_header(dicom_path)
            annotation_path = ANNOTATIONS_DIR / series_id / f"{header['sop_uid']}.json"
            segmentation_rle = None
            segmentation_valid = 0
            n_positive_ich_classes = np.nan

            if annotation_path.exists():
                with open(annotation_path, "r", encoding="utf-8") as file:
                    annotation = json.load(file)
                parsed = parse_rle_header(annotation.get("segmentation_rle"))
                if parsed is not None:
                    _, counts = parsed
                    labels_present = set(counts[0::2])
                    n_positive_ich_classes = len(labels_present & {1, 2, 3, 4, 5})
                    segmentation_rle = json.dumps(annotation["segmentation_rle"], separators=(",", ":"))
                    segmentation_valid = 1

            rows.append({
                "series_id": series_id,
                "split": split_name,
                "dicom_path": str(dicom_path),
                "sop_uid": header["sop_uid"],
                "row_spacing": header["row_spacing"],
                "col_spacing": header["col_spacing"],
                "instance_number": header["instance_number"],
                "position_scalar": header["position_scalar"],
                "slice_thickness": header["slice_thickness"],
                "spacing_between_slices": header["spacing_between_slices"],
                "segmentation_rle": segmentation_rle,
                "segmentation_valid": segmentation_valid,
                "n_positive_ich_classes": n_positive_ich_classes,
            })

    master_df = pd.DataFrame(rows)
    master_df.to_pickle(MASTER_INDEX_PATH)

master_df["series_id"] = master_df["series_id"].map(normalize_series_id)

print("Master slices:", len(master_df))
print("Valid segmentation slices:", int(master_df["segmentation_valid"].sum()))

## 9. Build expanded-negative TRAIN targets

In [ ]:
train_targets = train_series_df[["series_id"] + ICH_COLUMNS].copy()
dev_targets = dev_series_df[["series_id"] + ICH_COLUMNS].copy()
train_targets["series_true_total_ich"] = train_targets[ICH_COLUMNS].sum(axis=1)
dev_targets["series_true_total_ich"] = dev_targets[ICH_COLUMNS].sum(axis=1)
train_targets["series_true_any_ich"] = (train_targets["series_true_total_ich"] >= 0.1).astype(int)
dev_targets["series_true_any_ich"] = (dev_targets["series_true_total_ich"] >= 0.1).astype(int)

train_all = master_df[master_df["split"] == "TRAIN"].merge(train_targets[["series_id", "series_true_any_ich"]], on="series_id", how="left")
dev_all = master_df[master_df["split"] == "DEV"].merge(dev_targets[["series_id", "series_true_any_ich"]], on="series_id", how="left")

train_labeled_positive_series = train_all[(train_all["series_true_any_ich"] == 1) & (train_all["segmentation_valid"] == 1)].copy()
train_normal_series_all_slices = train_all[train_all["series_true_any_ich"] == 0].copy()
ich_train_df = pd.concat([train_labeled_positive_series, train_normal_series_all_slices], ignore_index=True)
ich_dev_df = dev_all[dev_all["segmentation_valid"] == 1].copy()

ich_train_df["synthetic_background_target"] = ((ich_train_df["series_true_any_ich"] == 0) & (ich_train_df["segmentation_valid"] == 0)).astype(int)
ich_dev_df["synthetic_background_target"] = 0

ich_train_df["slice_has_ich"] = (ich_train_df["n_positive_ich_classes"].fillna(0) > 0).astype(int)
ich_train_df["sample_category"] = np.select(
    [
        ich_train_df["slice_has_ich"] == 1,
        (ich_train_df["series_true_any_ich"] == 1) & (ich_train_df["slice_has_ich"] == 0),
        ich_train_df["series_true_any_ich"] == 0,
    ],
    ["positive_slice", "negative_in_positive_series", "expanded_negative_in_normal_series"],
    default="unknown",
)

if (ich_train_df["sample_category"] == "unknown").any():
    raise RuntimeError("Unknown training sample category detected.")
if ((ich_train_df["series_true_any_ich"] == 1) & (ich_train_df["segmentation_valid"] == 0)).any():
    raise RuntimeError("An unannotated slice from an ICH-positive series entered the training target table.")

audit = ich_train_df.groupby(["sample_category", "synthetic_background_target"]).size().reset_index(name="n_slices")
audit.to_csv(METRICS_DIR / "01_training_target_audit.csv", index=False)

print("Expanded training slices:", len(ich_train_df))
print("Synthetic background slices from fully ICH-negative series:", int(ich_train_df["synthetic_background_target"].sum()))
display(audit)

## 10. Image and mask preprocessing

In [ ]:
@lru_cache(maxsize=HU_CACHE_SIZE)
def load_hu_image(path):
    ds = pydicom.dcmread(path, force=True)
    return np.asarray(apply_modality_lut(ds.pixel_array, ds), dtype=np.float32)

def parse_rle(rle):
    if not isinstance(rle, dict) or "shape" not in rle or "counts" not in rle:
        return None
    shape = tuple(int(value) for value in rle["shape"])
    counts = [int(value) for value in rle["counts"]]
    if len(shape) != 2 or len(counts) % 2 != 0 or sum(counts[1::2]) != int(np.prod(shape)):
        return None
    return shape, counts

def decode_ich_mask(rle_json):
    parsed = parse_rle(json.loads(rle_json))
    if parsed is None:
        raise ValueError("Invalid segmentation RLE.")
    shape, counts = parsed
    original = np.empty(int(np.prod(shape)), dtype=np.uint8)
    position = 0
    for value, length in zip(counts[0::2], counts[1::2]):
        original[position:position + length] = value
        position += length
    original = original.reshape(shape)
    mask = np.zeros(shape, dtype=np.uint8)
    for new_id, subtype in enumerate(ICH_CLASSES, start=1):
        mask[original == ORIGINAL_CLASS_ID[subtype]] = new_id
    return torch.from_numpy(mask).long()

def blood_window(image_hu):
    low = BLOOD_WINDOW_CENTER - BLOOD_WINDOW_WIDTH / 2.0
    high = BLOOD_WINDOW_CENTER + BLOOD_WINDOW_WIDTH / 2.0
    return torch.from_numpy(((np.clip(image_hu, low, high) - low) / (high - low)).astype(np.float32))

def normalize_input(image):
    mean = torch.tensor(IMAGENET_MEAN, dtype=torch.float32).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=torch.float32).view(3, 1, 1)
    return (image - mean) / std

## 10.1. Build physical slice-order lookup for 2.5D context

In [ ]:
def sort_series_rows(dataframe):
    groups = {}
    lookup = {}

    for series_id, group in dataframe.groupby("series_id", sort=False):
        group = group.copy()

        if group["position_scalar"].notna().all():
            group = group.sort_values("position_scalar")
        elif group["instance_number"].notna().all():
            group = group.sort_values("instance_number")
        else:
            group = group.sort_values("dicom_path")

        group = group.reset_index(drop=True)
        groups[series_id] = group

        for index, row in group.iterrows():
            lookup[(series_id, str(row["sop_uid"]))] = index

    return groups, lookup

series_rows, center_lookup = sort_series_rows(master_df)

print("Indexed series:", len(series_rows))
print("Context offsets:", CONTEXT_OFFSETS)


## 11. 2.5D dataset with expanded true-negative targets

In [ ]:
class ExpandedNegative2p5DDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.df = dataframe.reset_index(drop=True)
        self.training = training
        self.geometric_transform = v2.Compose([v2.RandomHorizontalFlip(p=0.5), v2.RandomRotation(degrees=10)])

    def __len__(self):
        return len(self.df)

    def context_paths(self, row):
        group = series_rows[row["series_id"]]
        center = center_lookup[(row["series_id"], str(row["sop_uid"]))]
        indices = [min(max(center + offset, 0), len(group) - 1) for offset in CONTEXT_OFFSETS]
        return [group.iloc[index]["dicom_path"] for index in indices]

    def __getitem__(self, index):
        row = self.df.iloc[index]
        channels = [blood_window(load_hu_image(str(path))) for path in self.context_paths(row)]
        image = tv_tensors.Image(torch.stack(channels))

        if int(row["synthetic_background_target"]) == 1:
            mask = tv_tensors.Mask(torch.zeros(channels[1].shape, dtype=torch.long))
        else:
            mask = tv_tensors.Mask(decode_ich_mask(row["segmentation_rle"]))

        if self.training:
            image, mask = self.geometric_transform(image, mask)

        image = tv_tensors.Image(normalize_input(image))
        return image, mask.long(), row["series_id"], str(row["sop_uid"])


## 12. Controlled series-balanced and category-balanced sampling

In [ ]:
def build_sampler(dataframe):
    series_counts = dataframe.groupby("series_id").size().to_dict()
    weights = []

    for row in dataframe.itertuples(index=False):
        series_weight = 1.0 / series_counts[row.series_id]
        category_weight = CATEGORY_WEIGHT[row.sample_category]
        weights.append(series_weight * category_weight)

    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=int(SAMPLES_PER_EPOCH), replacement=True)

train_dataset = ExpandedNegative2p5DDataset(ich_train_df, training=True)
dev_dataset = ExpandedNegative2p5DDataset(ich_dev_df, training=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=build_sampler(ich_train_df), num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

print("TRAIN target slices available:", len(train_dataset))
print("Samples drawn per epoch:", SAMPLES_PER_EPOCH)
print("DEV labeled slices:", len(dev_dataset))

## 13. Model builder

In [ ]:
def model_slug():
    encoder = re.sub(r"[^a-zA-Z0-9]+", "-", MODEL_CONFIG["encoder_name"]).strip("-").lower()
    architecture = re.sub(r"[^a-zA-Z0-9]+", "-", MODEL_CONFIG["architecture"]).strip("-").lower()
    return f"ich_2p5d_blood_expanded_negatives_{architecture}_{encoder}"

def extract_state_dict(checkpoint):
    obj = torch.load(checkpoint, map_location="cpu")
    if isinstance(obj, dict) and "model_state_dict" in obj:
        return obj["model_state_dict"]
    if isinstance(obj, dict) and "state_dict" in obj:
        return obj["state_dict"]
    if isinstance(obj, dict):
        return obj
    raise TypeError(f"Unsupported checkpoint format: {type(obj)}")

def transfer_matching_weights(model, checkpoint_path):
    source = extract_state_dict(checkpoint_path)
    target = model.state_dict()
    matching = {key: value for key, value in source.items() if key in target and target[key].shape == value.shape}
    target.update(matching)
    model.load_state_dict(target)
    print(f"Transferred {len(matching)}/{len(target)} matching tensors from {checkpoint_path}.")

def build_model():
    builder = ARCHITECTURE_BUILDERS[MODEL_CONFIG["architecture"]]
    model = builder(encoder_name=MODEL_CONFIG["encoder_name"], encoder_weights=MODEL_CONFIG["encoder_weights"], in_channels=3, classes=6).to(DEVICE)

    if MODEL_CONFIG["custom_init_checkpoint"] is not None:
        checkpoint = Path(MODEL_CONFIG["custom_init_checkpoint"])
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
        transfer_matching_weights(model, checkpoint)

    return model

## 14. Loss and DEV Dice

In [ ]:
ce_loss = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True, classes=[1, 2, 3, 4, 5])

def criterion(logits, targets):
    ce = ce_loss(logits, targets)
    dice = dice_loss(logits, targets)
    return 0.5 * ce + 0.5 * dice

@torch.inference_mode()
def evaluate_dice(model, loader):
    model.eval()
    intersections = np.zeros(5, dtype=np.float64)
    denominators = np.zeros(5, dtype=np.float64)

    for images, masks, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        predictions = torch.argmax(model(images), dim=1)

        for class_id in range(1, 6):
            pred = predictions == class_id
            true = masks == class_id
            intersections[class_id - 1] += float((pred & true).sum().item())
            denominators[class_id - 1] += float(pred.sum().item() + true.sum().item())

    scores = {}
    valid = []
    for class_id, subtype in enumerate(ICH_CLASSES, start=1):
        denominator = denominators[class_id - 1]
        score = np.nan if denominator <= 0 else 2.0 * intersections[class_id - 1] / denominator
        scores[subtype] = float(score) if np.isfinite(score) else np.nan
        if np.isfinite(score):
            valid.append(score)

    return float(np.mean(valid)), scores

## 15. Train

In [ ]:
model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
best_path = MODELS_DIR / f"{model_slug()}_best.pth"
history_path = METRICS_DIR / "02_training_history.csv"
best_dice = -np.inf
best_epoch = None
epochs_without_improvement = 0
history = []

for epoch in range(EPOCHS):
    if epoch == LR_SWITCH_EPOCH:
        for group in optimizer.param_groups:
            group["lr"] = LR_AFTER_SWITCH

    model.train()
    running_loss = 0.0
    running_samples = 0
    bar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}", leave=False)

    for images, masks, _, _ in bar:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += float(loss.item()) * batch_size
        running_samples += batch_size
        bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / max(running_samples, 1)
    dev_mean_dice, dev_class_dice = evaluate_dice(model, dev_loader)
    row = {"epoch": epoch + 1, "train_loss": train_loss, "dev_mean_dice": dev_mean_dice, **{f"dev_dice_{key}": value for key, value in dev_class_dice.items()}}
    history.append(row)
    pd.DataFrame(history).to_csv(history_path, index=False)

    marker = ""
    if dev_mean_dice > best_dice:
        best_dice = dev_mean_dice
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_path)
        marker = " <-- Best"
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch + 1:02d} | loss {train_loss:.4f} | DEV Dice {dev_mean_dice:.4f}{marker}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE and epoch + 1 > LR_SWITCH_EPOCH:
        print(f"Early stopping after {epoch + 1} epochs.")
        break

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

print("Best epoch:", best_epoch)
print("Best DEV mean Dice:", round(best_dice, 4))

## 16. Tune class-confidence thresholds on DEV

In [ ]:
@torch.inference_mode()
def tune_thresholds(model, loader):
    intersections = {class_id: np.zeros(len(THRESHOLD_GRID), dtype=np.float64) for class_id in range(1, 6)}
    denominators = {class_id: np.zeros(len(THRESHOLD_GRID), dtype=np.float64) for class_id in range(1, 6)}

    for images, masks, _, _ in tqdm(loader, desc="Threshold tuning", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        probabilities = torch.softmax(model(images), dim=1)
        hard = torch.argmax(probabilities, dim=1)

        for class_id in range(1, 6):
            target = masks == class_id
            class_prob = probabilities[:, class_id]
            hard_class = hard == class_id

            for threshold_index, threshold in enumerate(THRESHOLD_GRID):
                pred = hard_class & (class_prob >= float(threshold))
                intersections[class_id][threshold_index] += float((pred & target).sum().item())
                denominators[class_id][threshold_index] += float(pred.sum().item() + target.sum().item())

    thresholds = {}
    rows = []

    for class_id, subtype in enumerate(ICH_CLASSES, start=1):
        dice = np.where(denominators[class_id] > 0, 2.0 * intersections[class_id] / denominators[class_id], 0.0)
        best_index = int(np.argmax(dice))
        thresholds[class_id] = float(THRESHOLD_GRID[best_index])
        rows.append({"class_id": class_id, "subtype": subtype, "threshold": thresholds[class_id], "dice": float(dice[best_index])})

    return thresholds, pd.DataFrame(rows)

thresholds, threshold_df = tune_thresholds(model, dev_loader)
threshold_df.to_csv(METRICS_DIR / "03_thresholds.csv", index=False)
display(threshold_df)

## 17. Full-series DEV inference

In [ ]:
def sort_series_rows(dataframe):
    groups = {}

    for series_id, group in dataframe.groupby("series_id", sort=False):
        group = group.copy()

        if group["position_scalar"].notna().all():
            group = group.sort_values("position_scalar")
        elif group["instance_number"].notna().all():
            group = group.sort_values("instance_number")
        else:
            group = group.sort_values("dicom_path")

        groups[series_id] = group.reset_index(drop=True)

    return groups

series_rows = sort_series_rows(master_df)

def derive_slice_depths_mm(series_table):
    positions = series_table["position_scalar"].to_numpy(dtype=np.float64)
    n = len(series_table)

    if n > 1 and np.all(np.isfinite(positions)):
        diffs = np.abs(np.diff(positions))
        positive = diffs[diffs > 1e-6]

        if len(positive):
            fallback = float(np.median(positive))
            diffs = np.where(diffs > 1e-6, diffs, fallback)
            depths = np.empty(n, dtype=np.float64)
            depths[0], depths[-1] = diffs[0], diffs[-1]

            if n > 2:
                depths[1:-1] = (diffs[:-1] + diffs[1:]) / 2.0

            return depths

    depths = []
    for row in series_table.itertuples(index=False):
        depth = float(row.spacing_between_slices)
        if not np.isfinite(depth) or depth <= 0:
            depth = float(row.slice_thickness)
        if not np.isfinite(depth) or depth <= 0:
            raise RuntimeError(f"Unable to determine slice depth for series {row.series_id}.")
        depths.append(depth)

    return np.asarray(depths, dtype=np.float64)

def probabilities_to_mask(probabilities):
    hard = torch.argmax(probabilities, dim=1)
    final = torch.zeros_like(hard)

    for class_id in range(1, 6):
        final[(hard == class_id) & (probabilities[:, class_id] >= thresholds[class_id])] = class_id

    return final

@torch.inference_mode()
def predict_series(series_id):
    group = series_rows[str(series_id)].copy()
    depths = derive_slice_depths_mm(group)
    volumes = {subtype: 0.0 for subtype in ICH_CLASSES}
    slice_records = []

    for start in range(0, len(group), INFERENCE_BATCH_SIZE):
        batch_rows = group.iloc[start:start + INFERENCE_BATCH_SIZE]
        images = []

        for local_row_index, row in enumerate(batch_rows.itertuples(index=False)):
            center_index = start + local_row_index
            context_indices = [min(max(center_index + offset, 0), len(group) - 1) for offset in CONTEXT_OFFSETS]
            channels = [blood_window(load_hu_image(str(group.iloc[index]["dicom_path"]))) for index in context_indices]
            images.append(normalize_input(torch.stack(channels)))

        probabilities = torch.softmax(model(torch.stack(images).to(DEVICE, non_blocking=True)), dim=1)
        predictions = probabilities_to_mask(probabilities).cpu().numpy()
        max_probabilities = probabilities.amax(dim=(2, 3)).cpu().numpy()

        for local_index, prediction in enumerate(predictions):
            index = start + local_index
            row = group.iloc[index]
            voxel_volume_ml = float(row["row_spacing"]) * float(row["col_spacing"]) * float(depths[index]) / 1000.0
            slice_record = {"series_id": str(series_id), "slice_order": index, "sop_uid": str(row["sop_uid"])}

            for class_id, subtype in enumerate(ICH_CLASSES, start=1):
                pixels = int(np.count_nonzero(prediction == class_id))
                volume = pixels * voxel_volume_ml
                volumes[subtype] += volume
                slice_record[f"pred_positive_{subtype}"] = int(pixels > 0)
                slice_record[f"pred_volume_{subtype}"] = float(volume)
                slice_record[f"max_probability_{subtype}"] = float(max_probabilities[local_index, class_id])

            slice_records.append(slice_record)

    result = {"series_id": str(series_id), **{f"V_{subtype}": float(volumes[subtype]) for subtype in ICH_CLASSES}}
    return result, pd.DataFrame(slice_records)

series_predictions = []
slice_predictions = []
runtime_rows = []

for series_id in tqdm(dev_series_df["series_id"], desc="DEV full-series inference"):
    started = time.perf_counter()
    series_result, slice_result = predict_series(series_id)
    runtime_rows.append({"series_id": series_id, "runtime_seconds": time.perf_counter() - started})
    series_predictions.append(series_result)
    slice_predictions.append(slice_result)

dev_pred_df = pd.DataFrame(series_predictions)
dev_slice_df = pd.concat(slice_predictions, ignore_index=True)
runtime_df = pd.DataFrame(runtime_rows)

dev_pred_df.to_csv(METRICS_DIR / "04_dev_series_predictions.csv", index=False)
dev_slice_df.to_csv(CACHE_DIR / "dev_slice_predictions.csv", index=False)
runtime_df.to_csv(METRICS_DIR / "05_dev_runtime.csv", index=False)

## 18. Series-level ICH metrics

In [ ]:
def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "FPR": float(fp / (tn + fp)) if tn + fp else np.nan,
    }

merged = dev_series_df.merge(dev_pred_df, on="series_id", suffixes=("_true", "_pred"))
true_total = merged[[f"{column}_true" for column in ICH_COLUMNS]].sum(axis=1)
pred_total = merged[[f"{column}_pred" for column in ICH_COLUMNS]].sum(axis=1)
presence = binary_metrics((true_total >= 0.1).astype(int), (pred_total >= 0.1).astype(int))

summary = {
    "best_DEV_mean_Dice": best_dice,
    "Any_ICH_F1": presence["F1"],
    "Any_ICH_precision": presence["precision"],
    "Any_ICH_recall": presence["recall"],
    "Any_ICH_specificity": presence["specificity"],
    "Any_ICH_FPR": presence["FPR"],
    "Any_ICH_FP": presence["FP"],
    "Any_ICH_FN": presence["FN"],
    "total_ICH_MAE_mL": float(mean_absolute_error(true_total, pred_total)),
    "mean_runtime_seconds": float(runtime_df["runtime_seconds"].mean()),
}

for subtype in ICH_CLASSES:
    column = f"V_{subtype}"
    summary[f"{subtype}_MAE_mL"] = float(mean_absolute_error(merged[f"{column}_true"], merged[f"{column}_pred"]))
    summary[f"{subtype}_presence_F1"] = float(f1_score((merged[f"{column}_true"] >= 0.1).astype(int), (merged[f"{column}_pred"] >= 0.1).astype(int), zero_division=0))

summary_df = pd.DataFrame([summary])
summary_df.to_csv(METRICS_DIR / "06_dev_ich_summary.csv", index=False)
display(summary_df)

## 19. False-positive continuity on true-negative DEV series

In [ ]:
def extract_run_lengths(group, positive_column):
    flags = group.sort_values("slice_order")[positive_column].astype(bool).to_numpy()
    runs = []
    start = None

    for index in range(len(flags) + 1):
        active = index < len(flags) and flags[index]

        if active and start is None:
            start = index

        if not active and start is not None:
            runs.append(index - start)
            start = None

    return runs

negative_dev_series = set(dev_series_df.loc[dev_series_df[ICH_COLUMNS].sum(axis=1) < 0.1, "series_id"])
negative_slices = dev_slice_df[dev_slice_df["series_id"].isin(negative_dev_series)].copy()
run_rows = []

for subtype in ICH_CLASSES:
    lengths = []

    for _, group in negative_slices.groupby("series_id"):
        lengths.extend(extract_run_lengths(group, f"pred_positive_{subtype}"))

    run_rows.append({
        "subtype": subtype,
        "n_false_positive_runs": len(lengths),
        "median_run_length": float(np.median(lengths)) if lengths else np.nan,
        "single_slice_run_rate": float(np.mean(np.asarray(lengths) == 1)) if lengths else np.nan,
        "run_le_2_rate": float(np.mean(np.asarray(lengths) <= 2)) if lengths else np.nan,
    })

run_df = pd.DataFrame(run_rows)
run_df.to_csv(METRICS_DIR / "07_false_positive_runs.csv", index=False)
display(run_df)

## 20. Fixed three-slice continuity diagnostic

In [ ]:
def filter_subtype_runs(slice_df, subtype, min_run):
    work = slice_df.copy()
    output_column = f"filtered_volume_{subtype}"
    work[output_column] = 0.0

    for _, group in work.groupby("series_id", sort=False):
        group = group.sort_values("slice_order")
        flags = group[f"pred_positive_{subtype}"].astype(bool).to_numpy()
        keep = np.zeros(len(group), dtype=bool)
        start = None

        for index in range(len(flags) + 1):
            active = index < len(flags) and flags[index]

            if active and start is None:
                start = index

            if not active and start is not None:
                if index - start >= min_run:
                    keep[start:index] = True
                start = None

        work.loc[group.index, output_column] = np.where(keep, group[f"pred_volume_{subtype}"], 0.0)

    return work

filtered_slices = dev_slice_df.copy()
for subtype in ICH_CLASSES:
    filtered_slices = filter_subtype_runs(filtered_slices, subtype, FIXED_CONTINUITY_MIN_RUN)

filtered_series = filtered_slices.groupby("series_id")[[f"filtered_volume_{subtype}" for subtype in ICH_CLASSES]].sum().reset_index()
filtered_series = filtered_series.rename(columns={f"filtered_volume_{subtype}": f"V_{subtype}" for subtype in ICH_CLASSES})
filtered_merged = dev_series_df.merge(filtered_series, on="series_id", suffixes=("_true", "_pred"))
filtered_true_total = filtered_merged[[f"{column}_true" for column in ICH_COLUMNS]].sum(axis=1)
filtered_pred_total = filtered_merged[[f"{column}_pred" for column in ICH_COLUMNS]].sum(axis=1)
filtered_presence = binary_metrics((filtered_true_total >= 0.1).astype(int), (filtered_pred_total >= 0.1).astype(int))

continuity_df = pd.DataFrame([{
    "continuity_min_run": FIXED_CONTINUITY_MIN_RUN,
    "Any_ICH_F1": filtered_presence["F1"],
    "Any_ICH_precision": filtered_presence["precision"],
    "Any_ICH_recall": filtered_presence["recall"],
    "Any_ICH_specificity": filtered_presence["specificity"],
    "Any_ICH_FPR": filtered_presence["FPR"],
    "Any_ICH_FP": filtered_presence["FP"],
    "Any_ICH_FN": filtered_presence["FN"],
    "total_ICH_MAE_mL": float(mean_absolute_error(filtered_true_total, filtered_pred_total)),
}])

continuity_df.to_csv(METRICS_DIR / "08_fixed_continuity_diagnostic.csv", index=False)
display(continuity_df)

## 21. Save TorchScript model and config

In [ ]:
torchscript_path = MODELS_DIR / f"{model_slug()}_torchscript.pt"
config_path = MODELS_DIR / f"{model_slug()}_config.json"

with torch.inference_mode():
    traced = torch.jit.trace(model.eval(), torch.zeros(1, 3, 512, 512, device=DEVICE))
traced.save(str(torchscript_path))

config = {
    "architecture": MODEL_CONFIG["architecture"],
    "encoder_name": MODEL_CONFIG["encoder_name"],
    "encoder_weights": MODEL_CONFIG["encoder_weights"],
    "input_semantics": "previous_center_next_blood",
    "context_offsets": list(CONTEXT_OFFSETS),
    "blood_window": [BLOOD_WINDOW_CENTER, BLOOD_WINDOW_WIDTH],
    "imagenet_mean": list(IMAGENET_MEAN),
    "imagenet_std": list(IMAGENET_STD),
    "class_names": ["Background"] + ICH_CLASSES,
    "class_confidence_thresholds": {str(key): float(value) for key, value in thresholds.items()},
    "expanded_true_negatives": True,
    "synthetic_background_only_when_series_true_ich_zero": True,
    "samples_per_epoch": SAMPLES_PER_EPOCH,
    "locked_test_used": False,
}

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

print("State dict:", best_path)
print("TorchScript:", torchscript_path)
print("Config:", config_path)

## 22. Direct answers

In [ ]:
direct_answers = pd.DataFrame([{
    "experiment": "2.5D Blood previous-center-next + expanded true negatives",
    "encoder": MODEL_CONFIG["encoder_name"],
    "best_DEV_mean_Dice": best_dice,
    "DEV_Any_ICH_F1": summary["Any_ICH_F1"],
    "DEV_Any_ICH_precision": summary["Any_ICH_precision"],
    "DEV_Any_ICH_recall": summary["Any_ICH_recall"],
    "DEV_Any_ICH_FPR": summary["Any_ICH_FPR"],
    "DEV_total_ICH_MAE_mL": summary["total_ICH_MAE_mL"],
    "DEV_mean_runtime_seconds": summary["mean_runtime_seconds"],
    "expanded_training_slices": len(ich_train_df),
    "synthetic_background_slices": int(ich_train_df["synthetic_background_target"].sum()),
}])

direct_answers.to_csv(OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv", index=False)
display(direct_answers)

print("Primary result:", OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv")
print("Training audit:", METRICS_DIR / "01_training_target_audit.csv")
print("ICH summary:", METRICS_DIR / "06_dev_ich_summary.csv")
print("False-positive runs:", METRICS_DIR / "07_false_positive_runs.csv")
print("Continuity diagnostic:", METRICS_DIR / "08_fixed_continuity_diagnostic.csv")

# What to compare with Notebook 16

The main comparison is:

- Notebook 16 improved 2.5D Blood baseline
- Notebook 17 improved 2.5D Blood with expanded true negatives

A useful expanded-negative result should lower false-positive rate and false-positive run count while preserving ICH recall and segmentation quality.

Do not choose the final model from slice Dice alone. The main target is series-level behavior.